# Lekce 13 - Paměť agenta s Cognee znalostními grafy


## Nastavení

Tento poznámkový blok ukazuje, jak vybudovat inteligentního **kódovacího asistenta** s přetrvávající pamětí pomocí [**Cognee**](https://www.cognee.ai/) znalostních grafů a **Microsoft Agent Frameworku** (MAF).

Cognee převádí nestrukturovaný text na strukturovaný, dotazovatelný znalostní graf podpořený vektorovými vloženími — což vašemu agentovi poskytuje bohatou, vztahově uvědomělou dlouhodobou paměť.

### Co se naučíte
1. **Vytvářet znalostní grafy**: Přeměnit profily vývojářů a osvědčené postupy na strukturované, dotazovatelné znalosti.
2. **Integrovat Cognee s MAF**: Použít funkce `@tool` k tomu, aby agent MAF mohl dotazovat znalostní graf Cognee.
3. **Konverzace s povědomím o relaci**: Udržovat kontext přes více otázek ve stejné relaci.
4. **Dlouhodobá paměť**: Uchovávat důležité znalosti přes relace a znovu je získávat v nových konverzacích.

### Požadavky
- Python 3.9+
- Redis běžící lokálně (`docker run -d -p 6379:6379 redis`) pro správu relací
- Klíč API pro LLM (např. OpenAI) — nastavit `LLM_API_KEY` v `.env`
- `CACHING=true` v `.env` (vyžadováno pro sezení Cognee)
- Projekt Microsoft Foundry s nasazeným chat modelem
- `AZURE_AI_PROJECT_ENDPOINT` a `AZURE_AI_MODEL_DEPLOYMENT_NAME` v `.env`
- Přihlášení do Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Typy agentní paměti

Tento notebook zkoumá stejné tři typy paměti jako hlavní notebook Lekce 13, ale používá Cognee jako backend pro dlouhodobou paměť:

| Typ paměti | Mechanismus | Doba života |
|---|---|---|
| **Pracovní** | `agent.create_session()` (MAF) | Jedna konverzace |
| **Krátkodobá** | Mezipaměť relace Cognee (Redis) | Jedna relace |
| **Dlouhodobá** | Cognee znalostní graf + vektory | Trvalá |

### Architektura paměti Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Připravte úložiště Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Část 1 — Vytváření znalostní báze

Zpracováváme tři typy dat, abychom vytvořili komplexní znalostní bázi pro našeho kódovacího asistenta:

1. **Profil vývojáře** — osobní odbornost a technické zázemí
2. **Nejlepší postupy v Pythonu** — Zen Pythonu s praktickými pokyny
3. **Historické konverzace** — minulé Q&A sezení mezi vývojáři a AI asistenty


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Vizualizace znalostního grafu

Cognee může vykreslit interaktivní HTML vizualizaci entit a vztahů, které extrahovalo.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Obohaťte paměť pomocí Memify

`memify()` analyzuje znalostní graf a vytváří inteligentní pravidla — identifikuje vzory, osvědčené postupy a vztahy mezi koncepty.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Část 2 — MAF agent s nástroji Cognee

Nyní vytvoříme MAF agenta, který může dotazovat znalostní graf Cognee pomocí funkcí `@tool`. To umožňuje agentovi využívat plnou sílu sémantického vyhledávání s povědomím o grafu a zároveň udržovat konverzační kontext prostřednictvím relací.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Pracovní paměť se sezeními

`AgentSession` (vytvořený pomocí `agent.create_session()`) poskytuje pracovní paměť v rámci sezení. Agent může odkazovat zpět na předchozí zprávy a zároveň dotazovat dlouhodobý znalostní graf Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Nová relace — Dlouhodobá paměť přetrvává

Zahájení nové relace vymaže pracovní paměť, ale znalostní graf Cognee je stále k dispozici. Agent může získat stejnou dlouhodobou znalost v úplně novém rozhovoru.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Shrnutí

V tomto sešitě jste vytvořili kódovacího asistenta, který kombinuje **pracovní paměť MAF** (`agent.create_session()`) s **dlouhodobým znalostním grafem Cognee**.

### Co jste se naučili
1. **Tvorba znalostního grafu**: Cognee zpracovává nestrukturovaný text a vytváří graf + vektorovou paměť.
2. **Obohacení grafu pomocí memify**: Odvozené fakta a bohatší vztahy nad vaším stávajícím grafem.
3. **Integrace MAF + Cognee**: Funkce `@tool` umožňují agentovi MAF přirozeně dotazovat graf Cognee.
4. **Pracovní paměť + dlouhodobá paměť**: `AgentSession` (přes `agent.create_session()`) poskytuje kontext relace, zatímco Cognee poskytuje trvalé znalosti.
5. **Filtrované vyhledávání pomocí NodeSets**: Cílení na specifické podmnožiny znalostního grafu (např. pouze principy).

### Hlavní poznatky
- **Cognee** přeměňuje surový text na strukturovanou, vztahově uvědomělou paměť – silnější než plochý vektorový obchod.
- **Funkce `@tool`** čistě propojují agenty MAF s externími znalostními systémy.
- **`AgentSession`** (přes `agent.create_session()`) udržuje kontext každého rozhovoru odděleně od dlouhodobých znalostí.
- Stejný znalostní graf slouží více relacím a agentům.

### Praktické aplikace
- **Vývojářští kopiloti**: Revize kódu, analýza incidentů, asistenti architektury
- **Kopiloti pro zákazníky**: Podpora agentů nad produktovou dokumentací, FAQ a poznámkami CRM
- **Interní odborní kopiloti**: Asistenti pro politiky, právo nebo bezpečnost s odůvodňováním na základě pokynů
- **Jednotné datové vrstvy**: Kombinace strukturovaných a nestrukturovaných dat do jednoho dotazovatelného grafu

### Další kroky
- Experimentujte s časovou uvědomělostí v Cognee
- Definujte ontologii OWL pro kvalitu grafu specifickou pro danou doménu
- Přidejte zpětnou vazbu uživatelů pro zlepšení vyhledávání v čase
- Rozšiřte na multiagentní systémy sdílející stejnou paměťovou vrstvu Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o omezení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). Přestože usilujeme o co největší přesnost, mějte prosím na paměti, že automatizované překlady mohou obsahovat chyby nebo nepřesnosti. Originální dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro kritické informace se doporučuje profesionální lidský překlad. Nejsme odpovědní za jakékoli nedorozumění nebo nesprávné interpretace vzniklé použitím tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
